# WO-09 — fixes before the demo page
Cell 2 writes the ten filled_profiles and Tristan da Cunha (`wo09_filled_profiles.md`). Cell 3 checks that "no cropland" and "no pasture" are said only of a true zero (item 2). Wording is `profile_templates.md`; rules are in `scripts/edop/whg/`. Results: `docs/design/dashboard/wo09_results.md`. Read-only except `wo09_filled_profiles.md`.

In [1]:
%matplotlib inline
# Cell 1 — Setup: wording file, server check, saved ring, flow and river tables
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C
from scripts.edop.whg import wo04_fill as W
from scripts.edop.whg import wo06_rings as R
from scripts.edop.whg import wo07_rivers as RV
from scripts.edop.whg import wo08_units as U8

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
FLOW = pd.read_csv(C.OUT / 'q2_flow_pairs.csv', dtype={'tag': str})
SAMPLE = C.population_points()
RING = R.load_table()
RIV = RV.load_table() if RV.RIVERS_CSV.exists() else None
print('commit', C.commit(), '| server:', W.get_json('/api/health'), '| wording:', W.templates_stamp(), '| ring table:', len(RING), 'basins | river table:', len(RIV) if RIV else 'not built yet', '| test places:', len(C.PLACES), '| sample points:', len(SAMPLE))

commit f153841 | server: {'status': 'ok'} | wording: profile_templates.md v19 (last commit f153841; edited since: yes); script wo04_fill.py (last commit f153841; edited since: no); script wo06_rings.py (last commit e516acb; edited since: yes); script wo07_rivers.py (last commit f153841; edited since: no); script wo07_classes.py (last commit 7090f1c; edited since: no); script wo08_units.py (last commit f153841; edited since: no) | ring table: 315 basins | river table: 574 | test places: 10 | sample points: 306


In [2]:
# Cell 2 — The ten test places and Tristan da Cunha: the filled_profiles (written to docs/design/dashboard/wo09_filled_profiles.md) and the sentences of the WO-09 units
OUTMD = ROOT / 'docs' / 'design' / 'dashboard' / 'wo09_filled_profiles.md'
t0 = time.perf_counter()
TEST = {}
parts = ['# WO-09: the dummy filled from real data, ten test places and one place with no basin (filled_profiles)', '',
         'Generated by `notebooks/edop/whg/wo09_fixes.ipynb` (commit ' + C.commit() + '). Every `yes` unit of `modal_dummy_annotated.md` is filled from data; each `maybe` or `no` unit is a placeholder in the form `⟦id maybe: needs ...⟧`. Units with nothing true to say are silent. Wording is the template file\'s, not polished.', '',
         'Wording from `' + W.templates_stamp() + '`.', '']
L = []
for name, (lat, lon) in C.PLACES.items():
    F = W.build_facts(lat, lon, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    Un = W.fill_units(F)
    parts.append(W.render_filled_profile(name, F, Un)); parts.append('---'); parts.append('')
    if F['pub'] is None:
        L.append(f'{name}: no basin'); continue
    TEST[name] = (F, Un)
    L.append(name)
    for u in ('S5.1a', 'S5.1b', 'S5.both_none', 'S8.2', 'S8.point', 'S8.osm', 'S9.2'):
        if u in Un:
            L.append(f'   {u}: ' + (Un[u]['text'] if Un[u]['ok'] else f'[silent: {Un[u]["why"]}]'))
OUTMD.write_text('\n'.join(parts))
print(f'wrote {OUTMD.name} | {W.templates_stamp()} | {time.perf_counter() - t0:.0f}s\n\n' + '\n'.join(L))

wrote wo09_filled_profiles.md | profile_templates.md v19 (last commit f153841; edited since: yes); script wo04_fill.py (last commit f153841; edited since: no); script wo06_rings.py (last commit e516acb; edited since: yes); script wo07_rivers.py (last commit f153841; edited since: no); script wo07_classes.py (last commit 7090f1c; edited since: no); script wo08_units.py (last commit f153841; edited since: no) | 17s

Timbuktu
   S5.1a: no cropland is recorded
   S5.1b: 28% of the basin is pasture
   S8.2: 3 measures are zero here and so have no percentile: Cropland extent, Karst area, Permafrost extent.
   S8.point: Elevation and distances are measured from the place's point in WHG, which may be the centre of a town rather than its shore or riverbank.
   S8.osm: River names contain information from OpenStreetMap, available under the ODbL.
   S9.2: [silent: the ecoregion's Wikipedia link is flagged as unreliable in wo05_wikilinks.csv; button hidden]
Tbilisi
   S5.1a: about 22% of the basin

In [ ]:
# Cell 3 — Item 2: over the 306 sample basins, how often "no cropland" or "no pasture" is said where the raw value in basin06 is above zero (about 5 to 10 minutes)
t0 = time.perf_counter()
rows = []
for k, s in enumerate(SAMPLE, 1):
    la, lo = float(s['lat']), float(s['lon'])
    F = W.build_facts(la, lo, conn, PAIRS, FLOW, ring=RING, rivers=RIV)
    if F['pub'] is None:
        continue
    Un = W.fill_units(F)
    raw = conn.execute('SELECT crp_pc_sse, pst_pc_sse FROM public.basin06 WHERE hybas_id = %s', (int(F['hid']),)).fetchone()
    for uid, v in (('S5.1a', raw[0]), ('S5.1b', raw[1])):
        u = Un[uid]
        said_none = u['ok'] and u['text'].startswith('no ')
        rows.append(dict(hid=int(F['hid']), unit=uid, raw=v, said_none=said_none, said=u['text'] if u['ok'] else None, both_none='S5.both_none' in Un and Un['S5.both_none']['ok']))
    if k % 100 == 0:
        print(f'  {k} done in {time.perf_counter() - t0:.0f}s')
N = pd.DataFrame(rows)
bad = N[N['said_none'] & (N['raw'].fillna(0) > 0)]
L = [f'sample basins checked: {N["hid"].nunique()} | units: {len(N)} | said "no ..." : {int(N["said_none"].sum())} | of those, raw value above zero: {len(bad)}',
     f'both cropland and pasture zero (S5.both_none): {int(N.groupby("hid")["both_none"].first().sum())} basins',
     f'{int((N["said"].fillna("").str.contains("of the basin is pasture")).sum())} basins say "of the basin is pasture" (cropland zero, pasture not)']
if len(bad):
    L.append('\n' + bad.to_string(index=False))
print('\n'.join(L) + f'\n\n{time.perf_counter() - t0:.0f}s')